In [0]:
import requests as requests
import os
from datetime import datetime, timezone

In [0]:
API_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
SOURCE = "VIIRS_SNPP_NRT"
AREA_SP = "-53.2,-25.4,-44.1,-19.7"  # Bounding Box de São Paulo
DAY_RANGE = 1

In [0]:
# Databricks storage configuration

CATALOG = "clima"

SCHEMA = "01_bronze"

VOLUME = "nasa_data"

OUTPUT_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
    "/raw_nasa"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [0]:
MAP_KEY = dbutils.secrets.get(
    scope="apinasakey", 
    key="nasakey"
    )

In [0]:
url = f"{API_URL}/{MAP_KEY}/{SOURCE}/{AREA_SP}/{DAY_RANGE}"

print(f"API configured for source: {SOURCE}")

print(f"Area: {AREA_SP}")

print(f"Day range: {DAY_RANGE}")

In [0]:
response = requests.get(url, timeout=120)

if response.status_code != 200:
    raise Exception(
        "NASA FIRMS API request failed. "
        f"Status code: {response.status_code}"
    )

print("NASA FIRMS API request successful.")

In [0]:
if response.text.strip():

    current_time = datetime.now(
        timezone.utc
    ).strftime("%Y%m%d_%H%M%S")

    file_name = (
        f"firms_sp_{current_time}.csv"
    )

    file_path = os.path.join(
        OUTPUT_PATH,
        file_name
    )

    with open(
        file_path,
        "w",
        encoding="utf-8"
    ) as file:
        file.write(response.text)

    print(
        f"Raw file saved successfully: "
        f"{file_path}"
    )

else:
    print("No data returned by NASA FIRMS API for the requested period.")